In [27]:
import warnings
warnings.filterwarnings("ignore")

Loading Dataset from Kagglehub

In [28]:
!pip install --user kagglehub
import sys
sys.path.append(r"C:\Users\fatab\AppData\Roaming\Python\Python313\site-packages")
import kagglehub
import os
import pandas as pd

# 1️⃣ Download dataset (returns folder path)
dataset_path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
print("✅ Dataset downloaded at:", dataset_path)

# 2️⃣ Check what files are inside
files = os.listdir(dataset_path)
print("Files in dataset folder:", files)

# 3️⃣ Find the CSV file
csv_file = None
for f in files:
    if f.endswith(".csv"):
        csv_file = os.path.join(dataset_path, f)
        break

if csv_file is None:
    raise FileNotFoundError("No CSV file found in the downloaded dataset!")

# 4️⃣ Load CSV with pandas
data = pd.read_csv(csv_file)
print(data.head())
print(data.info())


✅ Dataset downloaded at: C:\Users\fatab\.cache\kagglehub\datasets\mlg-ulb\creditcardfraud\versions\3
Files in dataset folder: ['creditcard.csv']
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -

Installing xgboost

In [29]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [30]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

Data Cleaning

In [31]:
# -----------------------
# 1️⃣ Check for missing values
# -----------------------
print("Missing values per column:\n", data.isnull().sum())

# -----------------------
# 2️⃣ Separate features and target
# -----------------------
X = data.drop('Class', axis=1)
y = data['Class']

# -----------------------
# 3️⃣ Scale 'Time' and 'Amount' columns
# -----------------------
scaler = StandardScaler()
X[['Time', 'Amount']] = scaler.fit_transform(X[['Time', 'Amount']])

# -----------------------
# 4️⃣ Split data into train and test sets
# -----------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}, {y_train.shape}")
print(f"Test set: {X_test.shape}, {y_test.shape}")

Missing values per column:
 Time      0
V1        0
V2        0
V3        0
V4        0
V5        0
V6        0
V7        0
V8        0
V9        0
V10       0
V11       0
V12       0
V13       0
V14       0
V15       0
V16       0
V17       0
V18       0
V19       0
V20       0
V21       0
V22       0
V23       0
V24       0
V25       0
V26       0
V27       0
V28       0
Amount    0
Class     0
dtype: int64
Training set: (227845, 30), (227845,)
Test set: (56962, 30), (56962,)


In [32]:
# -----------------------
# 5️⃣ Handle class imbalance with SMOTE (oversampling)
# -----------------------
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("After SMOTE:")
print("Normal transactions:", sum(y_train_res==0))
print("Fraud transactions:", sum(y_train_res==1))

After SMOTE:
Normal transactions: 227451
Fraud transactions: 227451


Training Model

In [33]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import joblib

# -----------------------
# 1️⃣ Train XGBoost Classifier
# -----------------------
model = XGBClassifier(
    use_label_encoder=False,   # suppress warning
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_res, y_train_res)

# -----------------------
# 2️⃣ Predict on test set
# -----------------------
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]  # probability for ROC-AUC

# -----------------------
# 3️⃣ Evaluation
# -----------------------
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_proba))

# -----------------------
# 4️⃣ Save the trained model
# -----------------------
joblib.dump(model, "xgb_creditcard_fraud_model.pkl")
print("\n✅ Model saved as 'xgb_creditcard_fraud_model.pkl'")


Confusion Matrix:
 [[56832    32]
 [   11    87]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.73      0.89      0.80        98

    accuracy                           1.00     56962
   macro avg       0.87      0.94      0.90     56962
weighted avg       1.00      1.00      1.00     56962

ROC-AUC Score: 0.9791588308086319

✅ Model saved as 'xgb_creditcard_fraud_model.pkl'
